## Secure Credentials Management

### Introduction: The Importance of Secure Credentials

Welcome to the first lesson of the Developer Security & Observability course. In this lesson, we will focus on secure credentials management using AWS Secrets Manager and the AWS SDK for Python (boto3).

Credentials are pieces of sensitive information, such as usernames, passwords, or API keys, that allow applications to access resources like databases or external services. If these credentials are not protected, they can be stolen or misused, leading to security breaches.

AWS provides tools to help you manage credentials safely:

* **AWS Secrets Manager** lets you store, manage, and retrieve secrets securely.
* **boto3** allows your applications to interact with AWS services using authenticated requests.

By the end of this lesson, you will know how to store a secret in AWS Secrets Manager and retrieve it securely in your application using boto3.

---

## Understanding AWS Credential Management

Before we dive in, let's understand how AWS credentials work with boto3.

When you use boto3 to interact with AWS services, it needs credentials to authenticate your requests. boto3 looks for credentials in several places, in this order:

1. Environment variables (`AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`)
2. AWS credentials file (`~/.aws/credentials`)
3. IAM roles (when running on AWS services like EC2, Lambda, ECS)
4. Other configured sources

In this practice environment, credentials are pre-configured for you. However, understanding the credential chain is important for deploying your applications securely in production.

> **Production Best Practice:** In production environments running on AWS infrastructure (like EC2 instances), you should use IAM roles instead of static credentials. IAM roles provide temporary credentials automatically, eliminating the need to store long-term access keys in your code or configuration files. We'll discuss this more at the end of the lesson.

---

## Storing Secrets with AWS Secrets Manager

Let's start by learning how to store a secret, such as database credentials, in AWS Secrets Manager using Python and the boto3 library.

### Step 1: Import Required Libraries

First, we need to import the necessary libraries. We will use `boto3` to interact with AWS and `json` to handle our secret data.

```python
import json
import boto3
from botocore.exceptions import ClientError
```

* `boto3` is the AWS SDK for Python. It allows us to interact with AWS services.
* `json` helps us format our secret data as a JSON string.
* `ClientError` is used for handling errors when making AWS API calls.

### Step 2: Create a Secrets Manager Client

Next, we create a client to interact with AWS Secrets Manager.

```python
sm = boto3.client('secretsmanager')
```

* This line creates a client object called `sm` that lets us call Secrets Manager functions.
* boto3 will automatically use the configured AWS credentials to authenticate this client.

### Step 3: Define the Secret

Let's define the secret we want to store. For example, database credentials:

```python
SECRET_NAME = "app/db/credentials"
payload = {
    "username": "appuser",
    "password": "S3cureP@ss!",
    "host": "db.example.com"
}
```

* `SECRET_NAME` is the name we will use to refer to this secret in AWS.
* `payload` is a dictionary containing our sensitive information.

### Step 4: Store the Secret

Now, let's store the secret in AWS Secrets Manager.

```python
resp = sm.create_secret(
    Name=SECRET_NAME,
    SecretString=json.dumps(payload)
)
print("Created:", resp["ARN"])
```

* `sm.create_secret` creates a new secret in AWS.
* `Name` is the name of the secret.
* `SecretString` is the secret data, converted to a JSON string.
* The response contains information about the created secret, including its ARN (Amazon Resource Name).

**Note on Secret Types:** AWS Secrets Manager supports two types of secrets:

* `SecretString` - Used for text-based secrets like JSON credentials (what we're using here)
* `SecretBinary` - Used for binary data like encryption keys or certificates

Most application secrets use `SecretString`, which is what we'll focus on in this lesson.

**About ARNs:** An ARN is AWS's unique identifier for resources. Think of it like a complete address that specifies exactly which secret you created, including the AWS account, region, and service. We print the ARN here for two reasons:

* **Confirmation** - It verifies your secret was created successfully
* **Future reference** - You can use the ARN to:
  * Grant specific permissions to this secret in IAM policies
  * Reference the exact secret in CloudFormation templates or other infrastructure code
  * Audit which secrets exist in your AWS account

While you can retrieve secrets using just the friendly name (like `"app/db/credentials"`), the ARN provides an unambiguous reference that includes all the context about where the secret lives.

**Example Output:**

```text
Created: arn:aws:secretsmanager:us-east-1:123456789012:secret:app/db/credentials-abc123
```

Now, your secret is safely stored in AWS Secrets Manager.

---

## Retrieving Secrets in Your Application

Once your secret is stored, you need a way for your application to retrieve it securely. boto3 makes this straightforward.

### Architecture Overview

Here's how the secure retrieval flow works:

```text
┌─────────────────────────────┐
│   Your Application          │
│   (Python + boto3)          │
│                             │
│  ┌───────────────────────┐  │
│  │ AWS Credentials       │  │
│  │ (configured via       │  │
│  │  boto3 credential     │  │
│  │  chain)               │  │
│  └───────────┬───────────┘  │
└──────────────┼──────────────┘
               │
               │ Authenticated Request
               │ (using credentials)
               ▼
    ┌─────────────────────┐
    │  AWS Secrets        │
    │  Manager            │
    │                     │
    │  • Stores secrets   │
    │  • Returns values   │
    └─────────────────────┘
```

**Key Points:**

* Your application uses boto3 to make authenticated requests
* boto3 automatically uses available AWS credentials (from the credential chain)
* Secrets Manager validates the credentials and returns the secret if authorized
* No secrets need to be stored in your application code

### Step 1: Define a Function to Retrieve the Secret

Let's write a function that fetches the secret from AWS Secrets Manager with basic error handling.

```python
def get_secret(name):
    try:
        val = sm.get_secret_value(SecretId=name)
        return json.loads(val["SecretString"])
    except ClientError as e:
        # Handle common errors
        error_code = e.response['Error']['Code']
        if error_code == 'ResourceNotFoundException':
            print(f"Error: Secret '{name}' not found")
        elif error_code == 'AccessDeniedException':
            print(f"Error: Access denied to secret '{name}'")
        else:
            print(f"Error retrieving secret: {error_code}")
        raise
```

* `get_secret` takes the name of the secret.
* `sm.get_secret_value` fetches the secret from AWS using the authenticated client.
* The secret is returned as a JSON string, so we use `json.loads` to convert it back to a Python dictionary.
* We catch `ClientError` to handle common issues like missing secrets or permission problems.
* Different error codes tell us what went wrong (secret doesn't exist, access denied, etc.).

### Step 2: Use the Function to Access the Secret

Now, let's use this function to get our database credentials and print the host.

```python
if __name__ == "__main__":
    secret = get_secret("app/db/credentials")
    print("Secret host:", secret["host"])
```

* This code retrieves the secret and prints the value of the `host` field.

**Example Output:**

```text
Secret host: db.example.com
```

---

## Critical Security Note: Logging Hygiene

⚠️ **NEVER log or print the actual secret values** (passwords, API keys, tokens) in your application logs. Notice in our example we only print `secret["host"]`, which is the database hostname - not sensitive information like the password.

**Do NOT do this:**

```python
# ❌ BAD - Logs sensitive data
print("Retrieved secret:", secret)
print("Password:", secret["password"])
logger.info(f"DB credentials: {secret}")
```

**Do this instead:**

```python
# ✅ GOOD - Only logs non-sensitive metadata
print("Secret retrieved successfully")
print("Connecting to host:", secret["host"])
logger.info("Database credentials loaded from Secrets Manager")
```

**Why is this important?**

* Application logs are often stored in monitoring systems, log files, or sent to third parties
* Logging secrets defeats the purpose of using Secrets Manager
* Leaked logs can expose credentials to unauthorized users

When debugging, you can verify secret retrieval succeeded without exposing the actual values.

---

## Production Deployment: Using IAM Roles on EC2

In this practice environment, credentials are pre-configured for learning purposes. However, in production, you should never hardcode credentials or rely on long-term access keys stored in configuration files.

**Best Practice: IAM Roles for EC2**

When deploying applications on AWS infrastructure like EC2 instances, you should:

1. Create an IAM role with permissions to access Secrets Manager
2. Attach the IAM role to your EC2 instance (called an instance profile)
3. Use the same boto3 code - no changes needed!

Here's how it works:

```text
┌─────────────────────────────┐
│      EC2 Instance           │
│  ┌───────────────────────┐  │
│  │   Your Application    │  │
│  │   (Python + boto3)    │  │
│  └───────────┬───────────┘  │
│              │              │
│  ┌───────────▼───────────┐  │
│  │      IAM Role         │  │
│  │  (Instance Profile)   │  │
│  │  Provides temporary   │  │
│  │  credentials via IMDS │  │
│  └───────────┬───────────┘  │
└──────────────┼──────────────┘
               │
               │ Authenticated Request
               │ (temporary credentials)
               ▼
    ┌─────────────────────┐
    │  AWS Secrets        │
    │  Manager            │
    └─────────────────────┘
```

**Key Advantages:**

* No static credentials in your code or configuration
* Automatic credential rotation - AWS provides temporary credentials that expire
* Same code - boto3 automatically discovers and uses the IAM role credentials
* Better security - credentials can't be leaked from configuration files

The boto3 code you've written in this lesson works identically whether using configured credentials (like in this practice environment) or IAM roles (like in production on EC2). The credential discovery is automatic!

---

## Summary and What's Next

In this lesson, you learned:

* Why it's important to keep credentials secure.
* How to use AWS Secrets Manager to store sensitive information.
* How to retrieve secrets using boto3 with authenticated AWS requests.
* Basic error handling when accessing secrets.
* The critical importance of not logging secret values in your application.
* The production best practice of using IAM roles on EC2 instead of static credentials.

You saw step by step how to create and store a secret, and how to access it securely from your application. In the next set of practice exercises, you will get hands-on experience with these concepts by writing and running code to manage secrets yourself. This will help you build secure applications on AWS from the start.

## Updating Database Connection Details

Now that you understand how to store and retrieve secrets with AWS Secrets Manager, let's practice working with the credential data structure itself. You'll modify an existing payload dictionary to include additional database connection information and update the password for better security.

Your task is to update the payload dictionary in `create_secret.py` with two changes:

* Add a new `port` field with the integer value `5432` (the default PostgreSQL port).
* Change the password from its current value to `NewS3cure123!`.

Keep all other code unchanged — you only need to update the payload dictionary. The other files will work automatically with your updated credentials.

**Note:** The code is designed to be idempotent, meaning you can run it multiple times safely. If the secret already exists, it will be updated instead of causing an error.

**Cleanup:** After completing this practice, it's good practice to delete the lab secret to prevent drift and potential charges. You can delete it using the AWS CLI:

```shell
aws secretsmanager delete-secret --secret-id app/db/credentials --recovery-window-in-days 7
```

This schedules deletion with a 7-day recovery window in case you need to recover the secret.

This hands-on practice will help you understand how different types of credential information can be structured and stored securely in AWS Secrets Manager.

```python
import json, boto3
sm = boto3.client('secretsmanager')
SECRET_NAME = "app/db/credentials"
# TODO: Add a "port" field with integer value 5432 and change the password to "NewS3cure123!"
payload = {"username":"appuser","password":"S3cureP@ss!","host":"db.example.com"}

if __name__ == "__main__":
    try:
        resp = sm.create_secret(Name=SECRET_NAME, SecretString=json.dumps(payload))
        print("Created:", resp["ARN"])
    except sm.exceptions.ResourceExistsException:
        resp = sm.update_secret(SecretId=SECRET_NAME, SecretString=json.dumps(payload))
        print("Updated:", resp["ARN"])
```

Here is the completed `create_secret.py` with `port` added and the password updated:

```python
import json, boto3
sm = boto3.client('secretsmanager')
SECRET_NAME = "app/db/credentials"
payload = {"username":"appuser","password":"NewS3cure123!","host":"db.example.com","port":5432}

if __name__ == "__main__":
    try:
        resp = sm.create_secret(Name=SECRET_NAME, SecretString=json.dumps(payload))
        print("Created:", resp["ARN"])
    except sm.exceptions.ResourceExistsException:
        resp = sm.update_secret(SecretId=SECRET_NAME, SecretString=json.dumps(payload))
        print("Updated:", resp["ARN"])
```

## Parsing JSON Secrets from AWS

Nice work learning how to store secrets in AWS Secrets Manager! Now, let's focus on the retrieval side and understand how data formats work when getting secrets back from AWS.

When AWS Secrets Manager returns a secret, it comes back as a JSON string, but your application code needs to access individual pieces like the host or password. This means you need to convert that JSON string into a Python dictionary that allows key-based access.

Your task is to fix the `get_secret` function in `use_secret_on_ec2.py`. Right now, the function returns the raw JSON string, but the main code expects a dictionary so it can access `secret["host"]`. Look for the TODO comment and add the missing line that parses the JSON string into a dictionary.

This exercise will teach you a key concept: data retrieved from external services often needs format conversion before your application can use it effectively.

**Note on Resource Cleanup:** In production environments, you should delete test secrets after use to avoid unnecessary charges. AWS Secrets Manager allows you to specify a `RecoveryWindowInDays` parameter (minimum 7 days) when deleting secrets, which provides a safety window for recovering accidentally deleted secrets.

```python
import json, boto3
sm = boto3.client('secretsmanager')

def get_secret(name):
    val = sm.get_secret_value(SecretId=name)
    # TODO: Add the line that converts the JSON string to a Python dictionary
    return val["SecretString"]

if __name__ == "__main__":
    secret = get_secret("app/db/credentials")
    print("Secret host:", secret["host"])
```

Here is the completed `use_secret_on_ec2.py` with the JSON string parsed into a dictionary using `json.loads`:

```python
import json, boto3
sm = boto3.client('secretsmanager')

def get_secret(name):
    val = sm.get_secret_value(SecretId=name)
    return json.loads(val["SecretString"])

if __name__ == "__main__":
    secret = get_secret("app/db/credentials")
    print("Secret host:", secret["host"])
```

## Using Constants for Secret Names

Excellent progress with data formatting! Now, let's tackle an important coding practice that becomes critical when managing secrets: using constants instead of hardcoded strings.

You'll notice that the code defines a `SECRET_NAME` constant at the top of `create_secret.py`, but then ignores it and uses a different hardcoded string when creating the secret. This creates a mismatch because other parts of the application expect to find the secret using the constant name.

Your task is to find where the secret is being created and replace the hardcoded string with the `SECRET_NAME` constant. When you run the code before fixing it, you'll see that it fails because the secret gets created with one name but is retrieved with another.

This exercise will teach you why consistency in naming is essential for maintainable and reliable applications.

**Cleanup Note:** After completing this practice, you can delete the test secret to avoid unnecessary charges ($0.40/month per secret). AWS Secrets Manager requires a recovery window when deleting secrets. You can delete it using:

```python
sm.delete_secret(SecretId=SECRET_NAME, RecoveryWindowInDays=7)
```

The `RecoveryWindowInDays` parameter (minimum 7, maximum 30) allows you to recover accidentally deleted secrets. For lab/test environments, using 7 days is appropriate.

```python
import json, boto3
sm = boto3.client('secretsmanager')
SECRET_NAME = "app/db/credentials"
payload = {"username":"appuser","password":"S3cureP@ss!","host":"db.example.com"}

if __name__ == "__main__":
    try:
        resp = sm.create_secret(Name="hardcoded-secret-name", SecretString=json.dumps(payload))
        print("Created:", resp["ARN"])
    except sm.exceptions.ResourceExistsException:
        resp = sm.update_secret(SecretId="hardcoded-secret-name", SecretString=json.dumps(payload))
        print("Updated:", resp["ARN"])
```

The hardcoded string `"hardcoded-secret-name"` appears in **two** places — the `create_secret` call and the `update_secret` fallback — so both need to be replaced with `SECRET_NAME` to keep the naming consistent everywhere. Here is the completed `create_secret.py`:

```python
import json, boto3
sm = boto3.client('secretsmanager')
SECRET_NAME = "app/db/credentials"
payload = {"username":"appuser","password":"S3cureP@ss!","host":"db.example.com"}

if __name__ == "__main__":
    try:
        resp = sm.create_secret(Name=SECRET_NAME, SecretString=json.dumps(payload))
        print("Created:", resp["ARN"])
    except sm.exceptions.ResourceExistsException:
        resp = sm.update_secret(SecretId=SECRET_NAME, SecretString=json.dumps(payload))
        print("Updated:", resp["ARN"])
```

## Handling Missing Secrets Gracefully

Perfect work with consistent naming practices! Now, let's address a crucial aspect of working with external services like AWS: error handling.

When your application tries to retrieve a secret that doesn't exist, AWS Secrets Manager will throw an error that can crash your program. This happens in real scenarios — maybe someone deleted the secret, or you made a typo in the secret name.

Your task is to add proper exception handling to the `get_secret` function in `use_secret_on_ec2.py`:

* Import `ClientError` from `botocore.exceptions` at the top of the file
* Wrap the `sm.get_secret_value()` call in a try-except block
* Catch `ClientError` exceptions and print `"Secret not found"`
* Return an empty dictionary `{}` when a secret is not found

The `main.py` file will test both a successful case and an error case to show you how your error handling works. This way, your application will handle missing secrets smoothly instead of crashing, making it much more reliable in production environments.

**Cleanup Note:** After completing the practice exercises, consider deleting the test secret to avoid unnecessary storage charges. You can delete a secret with a recovery window using:

```python
sm.delete_secret(SecretId=SECRET_NAME, RecoveryWindowInDays=7)
```

```python
import json, boto3
# TODO: Import ClientError from botocore.exceptions
sm = boto3.client('secretsmanager')

def get_secret(name):
    # TODO: Add try-except block to handle ClientError and return empty dict when secret not found
    val = sm.get_secret_value(SecretId=name)
    return json.loads(val["SecretString"])

if __name__ == "__main__":
    secret = get_secret("app/db/credentials")
    print("Secret host:", secret["host"])
```

Here is the completed `use_secret_on_ec2.py` with the import added and the try-except block wrapping the retrieval call:

```python
import json, boto3
from botocore.exceptions import ClientError
sm = boto3.client('secretsmanager')

def get_secret(name):
    try:
        val = sm.get_secret_value(SecretId=name)
        return json.loads(val["SecretString"])
    except ClientError:
        print("Secret not found")
        return {}

if __name__ == "__main__":
    secret = get_secret("app/db/credentials")
    print("Secret host:", secret["host"])
```

## Validating Retrieved Secret Data Integrity

Fantastic work mastering error handling for missing secrets! Now it's time to put everything together and build a complete, production-ready workflow that validates your data integrity.

You've learned how to create secrets, retrieve them, and handle errors when they don't exist. However, there's one more critical step: making sure the data you get back contains everything you expect. Just because you can retrieve a secret doesn't mean it has all the fields your application needs.

Your task is to modify `main.py` to add validation logic that checks the retrieved secret:

* Verify that all three required fields are present: `"username"`, `"password"`, and `"host"`
* Check that each field contains actual data (not empty strings)
* Print `"Secret validation successful - all fields present"` if everything looks good
* Print `"Validation failed: [fieldname] is missing or empty"` for any problematic field

This exercise will teach you to build reliable applications that verify their data at every step, which is essential for production systems that need to work correctly every time.

**Cleanup Note:** After completing this practice series, it's good practice to delete the test secret to avoid resource drift and potential charges. You can delete it using:

```python
sm.delete_secret(SecretId=SECRET_NAME, RecoveryWindowInDays=7)
```

The `RecoveryWindowInDays` parameter (minimum 7 days) allows you to recover the secret if deleted accidentally. For immediate deletion in test environments, you can add `ForceDeleteWithoutRecovery=True`.

```python
import json
from create_secret import sm, SECRET_NAME, payload
from use_secret_on_ec2 import get_secret

if __name__ == "__main__":
    try:
        resp = sm.create_secret(Name=SECRET_NAME, SecretString=json.dumps(payload))
        print("Created:", resp["ARN"])
    except sm.exceptions.ResourceExistsException:
        resp = sm.update_secret(SecretId=SECRET_NAME, SecretString=json.dumps(payload))
        print("Updated:", resp["ARN"])

    # Retrieve and use the secret
    secret = get_secret(SECRET_NAME)
    
    # TODO: Add validation logic to check that all required fields ("username", "password", "host") are present and non-empty
    # TODO: Print "Secret validation successful - all fields present" if all validations pass
    # TODO: Print "Validation failed: [fieldname] is missing or empty" if any field is missing or empty
```

Here is the completed `main.py` with the validation logic added:

```python
import json
from create_secret import sm, SECRET_NAME, payload
from use_secret_on_ec2 import get_secret

if __name__ == "__main__":
    try:
        resp = sm.create_secret(Name=SECRET_NAME, SecretString=json.dumps(payload))
        print("Created:", resp["ARN"])
    except sm.exceptions.ResourceExistsException:
        resp = sm.update_secret(SecretId=SECRET_NAME, SecretString=json.dumps(payload))
        print("Updated:", resp["ARN"])

    # Retrieve and use the secret
    secret = get_secret(SECRET_NAME)

    required_fields = ["username", "password", "host"]
    missing_fields = [field for field in required_fields if not secret.get(field)]

    if missing_fields:
        for field in missing_fields:
            print(f"Validation failed: {field} is missing or empty")
    else:
        print("Secret validation successful - all fields present")
```

`secret.get(field)` returns `None` when the key is absent and the actual value otherwise, so `not secret.get(field)` catches both a missing key and an empty-string value in one check — exactly the two failure cases the task calls out.